In [ ]:
import os
import json
import re
import torch
import librosa
import pandas as pd
from tqdm import tqdm
from transformers import AutoProcessor, VoxtralForConditionalGeneration
from peft import PeftModel
from sklearn.metrics import f1_score, accuracy_score
import wandb

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.8"
os.environ["TOKENIZERS_PARALLELISM"] = "true"
os.environ["WANDB_PROJECT"] = "Voxtral-GLaDOS-Multimodal"
os.environ["WANDB_LOG_MODEL"] = "true"
os.environ["WANDB_NOTEBOOK_NAME"] = "qlora_evaluation.ipynb"
wandb.login()
compute_dtype = torch.bfloat16
model_id = "mistralai/Voxtral-Mini-3B-2507"
device = "cuda" if torch.cuda.is_available() else "cpu"
lora_path = "./models/voxtral-glados-sft"

processor = AutoProcessor.from_pretrained(model_id)
base_model = VoxtralForConditionalGeneration.from_pretrained(
        model_id,
        # quantization_config=quantization_config, # commented out for loftq
        attn_implementation="flash_attention_2",
        device_map="cpu", # Use CPU for loftq
        low_cpu_mem_usage=True,
        dtype=compute_dtype
    )
model = PeftModel.from_pretrained(base_model, lora_path)
model.to(device)
model.eval()

In [ ]:
def extract_json_payloads(text):
    if not isinstance(text, str):
        return []
    payloads = []
    # Find all non-overlapping { ... } blocks
    matches = re.finditer(r'(\{.*?\})', text, re.DOTALL)
    for match in matches:
        try:
            payloads.append(json.loads(match.group(1)))
        except json.JSONDecodeError:
            pass
    return payloads

def parse_generated_text(text):
    parts = text.split('\n\n', 1)
    json_section = parts[0]
    text_response = parts[1] if len(parts) > 1 else ""
    pred_payloads = extract_json_payloads(json_section)
    return pred_payloads, text_response, json_section

def calculate_slot_f1_multi(true_payloads, pred_payloads):
    def get_slot_set(payloads):
        slots = set()
        for p in payloads:
            if not p: # Handle empty dictionary {}
                continue
            # Associate the slot with its specific service to prevent cross-intent collisions
            srv = p.get("service", "unknown")
            for k, v in p.items():
                if k != "service":
                    slots.add((srv, k, str(v)))
        return slots
    true_set = get_slot_set(true_payloads)
    pred_set = get_slot_set(pred_payloads)
    # If both true and predicted require NO slots (Information Request {}), it's a perfect match.
    if not true_set and not pred_set:
        return 1.0
    tp = len(true_set.intersection(pred_set))
    fp = len(pred_set - true_set)
    fn = len(true_set - pred_set)
    if (tp + fp) == 0 or (tp + fn) == 0:
        return 0.0
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    if (precision + recall) == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)

#### Load test dataset and evaluate

In [ ]:
# Load your test dataset (using a CSV structure similar to your training script)
test_df = pd.read_csv("data/combined_multimodal_dataset_test.csv")
results = []
# Define the exact text prompt used during training
SYSTEM_INSTRUCTION = "You are GLaDOS. Execute the spoken command, output the required JSON payload, and respond in character."
for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Evaluating Test Set"):
    audio_path = os.path.join("data/synthesized_train_16k/", row["Audio_File"])
    if not os.path.exists(audio_path):
        continue
    # Format the Input to match the Training Data exactly!
    conversation = [
        {"role": "user", "content": [
            {"type": "text", "text": SYSTEM_INSTRUCTION},
            {"type": "audio", "path": audio_path}
        ]}
    ]
    # Process Audio and Text
    inputs = processor.apply_chat_template(
        conversation,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)
    # Generate Output
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,
            temperature=0.2 # Low temperature for reliable JSON output
        )
    # Decode and Extract text (skipping the prompt tokens)
    input_len = inputs["input_ids"].shape[1]
    generated_text = processor.tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
    # Ground Truth Parsing (Handles "{} {}" safely)
    true_payloads = extract_json_payloads(row["Assistant_Payload"])
    # Prediction Parsing
    pred_payloads, pred_text_response, raw_json_section = parse_generated_text(generated_text)
    results.append({
        "audio_file": row["Audio_File"],
        "true_payloads": true_payloads,
        "pred_payloads": pred_payloads,
        "true_text": row["Target_GLaDOS_Response"],
        "pred_text_raw": generated_text
    })

#### Compute Metrics

In [ ]:
total_samples = len(results)
successful_parses = 0
correct_intents = 0
slot_f1_scores = []
strict_format_adherence = 0
for res in results:
    true_p = res["true_payloads"]
    pred_p = res["pred_payloads"]
    # Syntactic Validity (Did it output at least one JSON bracket structure?)
    if len(pred_p) > 0:
        successful_parses += 1
    # Intent Accuracy (Evaluated as a set to handle multiple actions)
    # E.g., ["lock.lock", "cover.close"] vs ["lock.lock", "cover.close"]
    # An empty payload {} will yield [None]
    true_intents = {p.get("service") for p in true_p}
    pred_intents = {p.get("service") for p in pred_p}
    if true_intents == pred_intents:
        correct_intents += 1
    # Slot F1-Score (Calculates across all JSONs; handles empty {} perfectly)
    slot_f1 = calculate_slot_f1_multi(true_p, pred_p)
    slot_f1_scores.append(slot_f1)
    # Format Discipline (No conversational filler BEFORE the JSON)
    if res["pred_text_raw"].lstrip().startswith("{"):
        strict_format_adherence += 1
# Final Computations
json_parse_rate = (successful_parses / total_samples) * 100 if total_samples > 0 else 0
intent_accuracy = (correct_intents / total_samples) * 100 if total_samples > 0 else 0
avg_slot_f1 = (sum(slot_f1_scores) / total_samples) * 100 if total_samples > 0 else 0
ifeval_strict_acc = (strict_format_adherence / total_samples) * 100 if total_samples > 0 else 0
print("=== Phase 3: Evaluation Metrics ===")
print(f"Total Test Samples: {total_samples}")
print(f"JSON Parse Rate (Syntactic Validity): {json_parse_rate:.2f}%")
print(f"Intent Accuracy (Multi-Intent / Info Request): {intent_accuracy:.2f}%")
print(f"Slot F1-Score (Semantic Accuracy): {avg_slot_f1:.2f}%")
print(f"IFEval Strict Accuracy (Format Discipline): {ifeval_strict_acc:.2f}%")
if (intent_accuracy + avg_slot_f1) > 0:
    slu_f1 = (2 * (intent_accuracy/100) * (avg_slot_f1/100)) / ((intent_accuracy/100) + (avg_slot_f1/100)) * 100
else:
    slu_f1 = 0.0
print(f"SLU-F1 (Unified semantic comprehension): {slu_f1:.2f}%")